# Hand landmark sequence collection

This notebook implements a data collection pipeline for hand gesture recognition using MediaPipe-based landmark detection. It captures sequences of hand landmarks from a webcam, processes them in real time, and stores them as structured datasets for later use in training machine learning models. The system guides the user through recording multiple labeled action sequences, ensuring consistency via countdowns and fixed-length buffers, while providing live visual feedback of detected hand landmarks.

In [1]:
import os
import pickle
from typing import Literal
import cv2
import time
import json
from landmarkers.inferences import Inference, InferenceSequence
from landmarkers.mp.hands import MPVideoLandmarker, MediapipeHandsMetadata
import numpy as np


with open('../config.json', 'r') as f:
	config = json.load(f)
common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = 500#common_configdsss
N_SEQUENCES = collect_config['n_sequences']
NUM_LANDMARKS = collect_config['num_landmarks']
COUNTDOWN = collect_config['countdown']
MODEL_PATH = collect_config['model_path']
NUM_HANDS = collect_config['num_hands']
WINDOW_WIDTH = collect_config['window_width']
WINDOW_HEIGHT = collect_config['window_height']


def create_empty_hand() -> Inference:
	zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
	return Inference(
		landmarks=zeros,
		world_landmarks=zeros,
		metadata=MediapipeHandsMetadata(category_name="Right", index=0, score=0.0)
	)


def get_hand(inferences, category_name: Literal['Left', 'Right']) -> Inference:
	if not inferences:
		return create_empty_hand()
	hands = [inf for inf in inferences if inf.metadata.category_name == category_name]
	if hands:
		return hands[0]
	else:
		zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
		return Inference(
			landmarks=zeros,
			world_landmarks=zeros,
			metadata=MediapipeHandsMetadata(category_name=category_name, index=0, score=0.0)
		)


def draw_landmarks(frame, hand_right, hand_left):
	for lm in hand_right.landmarks.array:
		x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
		cv2.circle(frame, (x, y), 3, (0,255,0), -1)

	for lm in hand_left.landmarks.array:
		x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
		cv2.circle(frame, (x, y), 3, (0,0,255), -1)


def process_frame(landmarker, frame):
	ts = int(time.time() * 1000)
	inferences = landmarker.infer(frame, ts)
	hand_right = get_hand(inferences, "Right")
	hand_left = get_hand(inferences, "Left")
	return hand_right, hand_left, ts


def wait_for_start(cap, landmarker, window_name, action, seq_idx):
	print(f"\nReady to record: {action} sequence {seq_idx}. Press 's' to start.")

	while True:
		ret, frame = cap.read()
		if not ret:
			continue

		hand_right, hand_left, _ = process_frame(landmarker, frame)
		draw_landmarks(frame, hand_right, hand_left)

		cv2.putText(frame, f"Press 's' to start {action}{seq_idx}", (10,50),
					cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
		cv2.imshow(window_name, frame)

		key = cv2.waitKey(10) & 0xFF
		if key == ord('s'):
			return True
		elif key == ord('q'):
			return False


def run_countdown(cap, landmarker, window_name):
	start_time = time.time()

	while True:
		elapsed = time.time() - start_time
		remaining = COUNTDOWN - int(elapsed)
		if remaining <= 0:
			break

		ret, frame = cap.read()
		if not ret:
			continue

		hand_right, hand_left, _ = process_frame(landmarker, frame)
		draw_landmarks(frame, hand_right, hand_left)

		cv2.putText(frame, f"Starting in {remaining}...", (10,50),
					cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
		cv2.imshow(window_name, frame)
		cv2.waitKey(1)

		if int(elapsed) != int(elapsed - 0.05):
			print(f"{remaining}...")

	print("Recording now!")


def capture_sequence(cap, landmarker, window_name, action, seq_idx, sequence_length):
	# seq_right = InferenceSequence(fixed_buffer_length=sequence_length)
	# seq_left = InferenceSequence(fixed_buffer_length=sequence_length)
	seq_right = InferenceSequence()
	seq_left = InferenceSequence()

	frames_captured = 0

	# while frames_captured < sequence_length:
	while ((seq_right.time_stamps_ms[-1] if seq_right.time_stamps_ms else 0) - (seq_right.time_stamps_ms[0] if seq_right.time_stamps_ms else 0)) < sequence_length:
		ret, frame = cap.read()
		if not ret:
			continue

		hand_right, hand_left, ts = process_frame(landmarker, frame)

		seq_right.append(hand_right, ts)
		seq_left.append(hand_left, ts)

		draw_landmarks(frame, hand_right, hand_left)

		# cv2.putText(frame,
		#             f"{action}{seq_idx} frame {frames_captured+1}/{sequence_length}",
		#             (10,80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
		cv2.putText(frame,
			f"{action}{seq_idx} time left {(sequence_length - ((seq_right.time_stamps_ms[-1] if seq_right.time_stamps_ms else 0) - (seq_right.time_stamps_ms[0] if seq_right.time_stamps_ms else 0)))}",
			(10,80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

		cv2.imshow(window_name, frame)

		frames_captured += 1
		if cv2.waitKey(1) & 0xFF == ord('q'):
			return None, None

	print(f"frames captured: {frames_captured}")
	return seq_right, seq_left


def save_sequence(seq_right, seq_left, action, seq_idx):
	seq_folder = os.path.join(DATA_PATH, action, f"seq_{seq_idx}")
	os.makedirs(seq_folder, exist_ok=True)

	with open(os.path.join(seq_folder, "right.pkl"), "wb") as f:
		pickle.dump(seq_right, f)

	with open(os.path.join(seq_folder, "left.pkl"), "wb") as f:
		pickle.dump(seq_left, f)

	print(f"Sequence saved in {seq_folder}")


def record_sequence(landmarker, cap, action, seq_idx, sequence_length):
	window_name = f"{action}{seq_idx}"

	ret, frame = cap.read()
	if not ret:
		return False

	cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
	cv2.resizeWindow(window_name, WINDOW_WIDTH, WINDOW_HEIGHT)

	if not wait_for_start(cap, landmarker, window_name, action, seq_idx):
		cv2.destroyWindow(window_name)
		return False

	run_countdown(cap, landmarker, window_name)

	seq_right, seq_left = capture_sequence(
		cap, landmarker, window_name, action, seq_idx, sequence_length
	)

	if seq_right is None:
		cv2.destroyWindow(window_name)
		return False

	save_sequence(seq_right, seq_left, action, seq_idx)

	cv2.destroyWindow(window_name)
	return True


def record_actions(landmarker, cap):
	for action in ACTIONS:
		for seq_idx in range(N_SEQUENCES):
			success = record_sequence(landmarker, cap, action, seq_idx, SEQUENCE_LENGTH)
			if not success:
				print("Recording interrupted by user")
				return


def main():
	os.makedirs(DATA_PATH, exist_ok=True)
	for action in ACTIONS:
		os.makedirs(os.path.join(DATA_PATH, action), exist_ok=True)
	
	cap = cv2.VideoCapture(0)
	try:
		with MPVideoLandmarker(model_path=MODEL_PATH, num_hands=NUM_HANDS) as landmarker:
			print("Starting sequence recording...")
			record_actions(landmarker, cap)
	finally:
		cap.release()
		cv2.destroyAllWindows()
		print("Recording finished")


if __name__ == "__main__":
	main()

2026-04-24 15:56:25.355414: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-24 15:56:25.990507: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-24 15:56:28.258083: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777038990.541817   41141 inference_feedback_manager.cc:114] Feedback

Starting sequence recording...



Ready to record: NONE sequence 0. Press 's' to start.


W0000 00:00:1777038991.788007   41142 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


1...
Recording now!
frames captured: 13
Sequence saved in ./dataset/NONE/seq_0

Ready to record: NONE sequence 1. Press 's' to start.
1...
Recording now!
frames captured: 14
Sequence saved in ./dataset/NONE/seq_1

Ready to record: NONE sequence 2. Press 's' to start.
1...
Recording now!
frames captured: 14
Sequence saved in ./dataset/NONE/seq_2

Ready to record: NONE sequence 3. Press 's' to start.
1...
Recording now!
frames captured: 13
Sequence saved in ./dataset/NONE/seq_3

Ready to record: NONE sequence 4. Press 's' to start.
1...
1...
Recording now!
frames captured: 12
Sequence saved in ./dataset/NONE/seq_4

Ready to record: NONE sequence 5. Press 's' to start.
1...
1...
Recording now!
frames captured: 14
Sequence saved in ./dataset/NONE/seq_5

Ready to record: NONE sequence 6. Press 's' to start.
1...
Recording now!
frames captured: 15
Sequence saved in ./dataset/NONE/seq_6

Ready to record: NONE sequence 7. Press 's' to start.
1...
Recording now!
frames captured: 12
Sequence sav